# 05 — Tune and freeze the joint allocator on validation only

The joint score is `J_i = alpha * norm(P_i) - (1-alpha) * norm(D_i)`. `alpha` is selected using an **auxiliary image-level detector** trained on train images and scored on validation stegos at one frozen payload level. No test image is used. The selected `alpha` is written to `config/frozen_allocator.json`.

In [1]:
from pathlib import Path
import json, joblib, pandas as pd, numpy as np, yaml
from rdhlab.io import read_gray
from rdhlab.pipeline import (load_payload_freeze, manifest_sha256, run_frozen_image, fit_aux_image_detector, score_aux_image_detector)

config=yaml.safe_load(Path('/workspace/config/experiment.yaml').read_text())
seed=int(config['project']['seed']); bs=int(config['dataset']['block_size'])
manifest_path=Path(config['dataset']['prepared_manifest']); manifest=pd.read_csv(manifest_path)
train=manifest[manifest.split=='train'].reset_index(drop=True); val=manifest[manifest.split=='validation'].reset_index(drop=True)
risk=joblib.load('/workspace/results/models/block_risk.joblib')
payload_freeze=load_payload_freeze('/workspace/config/frozen_payloads.json')
levels=list(map(float,payload_freeze['levels']))
idx=min(int(config['allocator']['tune_payload_index']),len(levels)-1)
tune_bpp=levels[idx]
print('Frozen payload levels:',levels,'tuning at',tune_bpp,'bpp')

Frozen payload levels: [0.003, 0.006, 0.009, 0.012] tuning at 0.006 bpp


In [2]:
# Train a deliberately simple auxiliary image detector on a method mixture.
strategies=['raster','random','predictability','detectability']
n_train=min(int(config['allocator']['auxiliary_train_images']),len(train))
covers=[]; stegos=[]
for j,row in train.head(n_train).iterrows():
    x=read_gray(row.path); strategy=strategies[j%len(strategies)]
    rr=run_frozen_image(x,str(row.source_id),tune_bpp,strategy,risk,.5,bs,seed)
    if rr['feasible']:
        covers.append(x); stegos.append(rr['stego'])
    if (j+1)%100==0: print('aux train',j+1,'/',n_train)
aux=fit_aux_image_detector(covers,stegos,seed)
joblib.dump(aux,'/workspace/results/models/aux_image_detector.joblib')
print('Auxiliary detector pairs:',len(covers))

aux train 100 / 1500
aux train 200 / 1500
aux train 300 / 1500
aux train 400 / 1500
aux train 500 / 1500
aux train 600 / 1500
aux train 700 / 1500
aux train 800 / 1500
aux train 900 / 1500
aux train 1000 / 1500
aux train 1100 / 1500
aux train 1200 / 1500
aux train 1300 / 1500
aux train 1400 / 1500
aux train 1500 / 1500
Auxiliary detector pairs: 1498


In [3]:
alpha_rows=[]
val_n=min(int(config['allocator']['validation_images']),len(val))
for alpha in map(float,config['allocator']['alpha_grid']):
    scores=[]; psnrs=[]; feasible=0
    for j,row in val.head(val_n).iterrows():
        x=read_gray(row.path)
        rr=run_frozen_image(x,str(row.source_id),tune_bpp,'joint',risk,alpha,bs,seed)
        if rr['feasible']:
            feasible+=1; scores.append(score_aux_image_detector(aux,rr['stego'])); psnrs.append(rr['psnr'])
        if (j+1)%250==0: print('alpha',alpha,j+1,'/',val_n)
    alpha_rows.append({'alpha':alpha,'n':val_n,'feasible_fraction':feasible/val_n,'aux_score_mean':np.mean(scores),'aux_score_median':np.median(scores),'psnr_mean':np.mean(psnrs)})
tuning=pd.DataFrame(alpha_rows)
tuning.to_csv('/workspace/results/models/allocator_alpha_validation.csv',index=False)
display(tuning)

alpha 0.0 250 / 2000
alpha 0.0 500 / 2000
alpha 0.0 750 / 2000
alpha 0.0 1000 / 2000
alpha 0.0 1250 / 2000
alpha 0.0 1500 / 2000
alpha 0.0 1750 / 2000
alpha 0.0 2000 / 2000
alpha 0.25 250 / 2000
alpha 0.25 500 / 2000
alpha 0.25 750 / 2000
alpha 0.25 1000 / 2000
alpha 0.25 1250 / 2000
alpha 0.25 1500 / 2000
alpha 0.25 1750 / 2000
alpha 0.25 2000 / 2000
alpha 0.5 250 / 2000
alpha 0.5 500 / 2000
alpha 0.5 750 / 2000
alpha 0.5 1000 / 2000
alpha 0.5 1250 / 2000
alpha 0.5 1500 / 2000
alpha 0.5 1750 / 2000
alpha 0.5 2000 / 2000
alpha 0.75 250 / 2000
alpha 0.75 500 / 2000
alpha 0.75 750 / 2000
alpha 0.75 1000 / 2000
alpha 0.75 1250 / 2000
alpha 0.75 1500 / 2000
alpha 0.75 1750 / 2000
alpha 0.75 2000 / 2000
alpha 1.0 250 / 2000
alpha 1.0 500 / 2000
alpha 1.0 750 / 2000
alpha 1.0 1000 / 2000
alpha 1.0 1250 / 2000
alpha 1.0 1500 / 2000
alpha 1.0 1750 / 2000
alpha 1.0 2000 / 2000


,alpha,n,feasible_fraction,aux_score_mean,aux_score_median,psnr_mean
0,0.00,2000,0.9935,0.499708,0.499728,58.489488
1,0.25,2000,0.9935,0.499708,0.499733,59.541878
2,0.50,2000,0.9935,0.499708,0.499736,60.894016
3,0.75,2000,0.9935,0.499708,0.499743,61.806977
4,1.00,2000,0.9935,0.499707,0.499736,62.301634


In [5]:
eligible=tuning[tuning.feasible_fraction>=float(config['payload']['minimum_validation_feasibility'])].copy()
if eligible.empty: raise RuntimeError('No alpha meets the predeclared validation feasibility threshold.')
eligible['tie_distance']=abs(eligible.alpha-.5)
best=eligible.sort_values(['aux_score_mean','psnr_mean','tie_distance'],ascending=[True,False,True]).iloc[0]
freeze={
  'experiment_version':config['project']['version'], 'manifest_sha256':manifest_sha256(manifest_path),
  'split_used_for_tuning':'validation','tune_net_bpp':tune_bpp,'alpha':float(best.alpha),'beta':float(1-best.alpha),
  'selection_metric':'minimum mean auxiliary stego probability; PSNR and distance to 0.5 are tie-breakers',
  'alpha_grid':list(map(float,config['allocator']['alpha_grid']))
}
fp=Path('/workspace/config/frozen_allocator.json')
if fp.exists():
    old=json.loads(fp.read_text()); assert old['manifest_sha256']==freeze['manifest_sha256']; print('Existing freeze kept:',old)
else:
    fp.write_text(json.dumps(freeze,indent=2,sort_keys=True)); print('FROZEN:',freeze)

Existing freeze kept: {'alpha': 1.0, 'alpha_grid': [0.0, 0.25, 0.5, 0.75, 1.0], 'beta': 0.0, 'experiment_version': '0.2.0', 'manifest_sha256': '2ea33fc686971261b78184838853ad6313e660d566faf929e247b5251c9ab65f', 'selection_metric': 'minimum mean auxiliary stego probability; PSNR and distance to 0.5 are tie-breakers', 'split_used_for_tuning': 'validation', 'tune_net_bpp': 0.006}


After this notebook, both payload levels and allocator weight are frozen. Notebook 06 is the first notebook that evaluates the proposed strategies on the test split.